Loading the cleaned data

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

data_dir = Path("../data")
if not data_dir.exists():
    data_dir = Path("data")

df = pd.read_csv(data_dir / "hotel_bookings_cleaned.csv")
print(df.shape)  

(118563, 30)


Arrival date (need for booking_month)

In [2]:
df["arrival_date"] = pd.to_datetime(
    df["arrival_date_year"].astype(str) + "-" +
    df["arrival_date_month"] + "-" +
    df["arrival_date_day_of_month"].astype(str),
    format="%Y-%B-%d"
)

Engineered features

In [3]:
# total_stay and total_guests
df["total_stay"] = df["stays_in_week_nights"] + df["stays_in_weekend_nights"]
df["total_guests"] = df["adults"] + df["children"] + df["babies"]

# booking_lead_category: short 0-30, medium 31-120, long 121+
df["booking_lead_category"] = pd.cut(
    df["lead_time"],
    bins=[-1, 30, 120, np.inf],
    labels=["short", "medium", "long"]
)

# booking_category from has_agent / has_company
# the 216 bookings with both are counted as "agent"
# df["booking_category"] = np.select(
#     [df["has_agent"], df["has_company"]],
#     ["agent", "company"],
#     default="individual"
# )

df["booking_category"] = "individual"
df["booking_category"] = df["booking_category"].where(~df["has_company"], "company")
df["booking_category"] = df["booking_category"].where(~df["has_agent"], "agent") 

# # historical_cancellation_rate, with no division by 0
# prev_total = df["previous_cancellations"] + df["previous_bookings_not_canceled"]
# df["has_booking_history"] = (prev_total > 0).astype(int)
# df["historical_cancellation_rate"] = np.where(
#     prev_total > 0,
#     df["previous_cancellations"] / prev_total.replace(0, np.nan),
#     0.0
# )

prev_total = df["previous_cancellations"] + df["previous_bookings_not_canceled"]

# 0 / 0 gives NaN (no history), so fill those with 0 → no division-by-zero error
df["historical_cancellation_rate"] = (df["previous_cancellations"] / prev_total).fillna(0)
df["has_booking_history"] = (prev_total > 0).astype(int)

# booking_month: arrival date minus lead time
df["booking_date"] = df["arrival_date"] - pd.to_timedelta(df["lead_time"], unit="D")
df["booking_month"] = df["booking_date"].dt.month

display(df[["total_stay", "total_guests", "lead_time", "booking_lead_category",
            "booking_category", "historical_cancellation_rate", "booking_month"]].head(100))

,total_stay,total_guests,lead_time,booking_lead_category,booking_category,historical_cancellation_rate,booking_month
0,1,1,7,short,individual,0.0,6
1,1,1,13,short,agent,0.0,6
2,2,2,14,short,agent,0.0,6
3,2,2,14,short,agent,0.0,6
4,2,2,0,short,individual,0.0,7
...,...,...,...,...,...,...,...
95,7,3,63,medium,agent,0.0,5
96,7,3,101,medium,agent,0.0,3
97,7,2,102,medium,agent,0.0,3
98,3,2,4,short,agent,0.0,6


In [4]:
for col in ["booking_lead_category", "booking_category", "has_booking_history", "booking_month"]:
    summary = df.groupby(col, observed=True)["is_canceled"].agg(Count="size", Cancel_Rate="mean")
    summary["Cancel_Rate"] = (summary["Cancel_Rate"] * 100).round(2)
    display(summary)

# categorical values consistent?
for col in ["hotel", "meal", "market_segment", "distribution_channel",
            "deposit_type", "customer_type", "booking_category"]:
    print(col, sorted(df[col].unique()))

,Count,Cancel_Rate
booking_lead_category,,
short,38094,18.77
medium,39850,39.60
long,40619,52.30


,Count,Cancel_Rate
booking_category,,
agent,102484,39.19
company,6479,17.87
individual,9600,29.68


,Count,Cancel_Rate
has_booking_history,,
0,109147,34.97
1,9416,63.82


,Count,Cancel_Rate
booking_month,,
1,16600,35.74
2,13395,35.89
3,9844,32.52
4,8577,35.56
5,7775,32.80
6,6008,35.52
7,9770,44.34
8,7913,30.66
9,8128,32.63


hotel ['City Hotel', 'Resort Hotel']
meal ['BB', 'FB', 'HB', 'SC']
market_segment ['Aviation', 'Complementary', 'Corporate', 'Direct', 'Groups', 'Offline TA/TO', 'Online TA', 'Undefined']
distribution_channel ['Corporate', 'Direct', 'GDS', 'TA/TO', 'Undefined']
deposit_type ['No Deposit', 'Non Refund', 'Refundable']
customer_type ['Contract', 'Group', 'Transient', 'Transient-Party']
booking_category ['agent', 'company', 'individual']


In [5]:
# df = df.sort_values("arrival_date", kind="stable").reset_index(drop=True)

# # cut on whole dates so the same day never ends up in two splits
# cut_train = df["arrival_date"].iloc[int(len(df) * 0.70)]
# cut_val   = df["arrival_date"].iloc[int(len(df) * 0.85)]

# train = df[df["arrival_date"] < cut_train]
# val   = df[(df["arrival_date"] >= cut_train) & (df["arrival_date"] < cut_val)]
# test  = df[df["arrival_date"] >= cut_val]

# split_summary = pd.DataFrame({
#     "Rows": [len(train), len(val), len(test)],
#     "Share (%)": [round(len(s) / len(df) * 100, 1) for s in (train, val, test)],
#     "From": [s["arrival_date"].min().date() for s in (train, val, test)],
#     "To":   [s["arrival_date"].max().date() for s in (train, val, test)],
#     "Cancellation rate (%)": [round(s["is_canceled"].mean() * 100, 2) for s in (train, val, test)],
# }, index=["Train", "Validation", "Test"])
# display(split_summary)

# Lab 4 shuffled with sample(frac=1). For time order we sort instead, then cut the same way.
df = df.sort_values("arrival_date").reset_index(drop=True)

cut1 = int(0.70 * len(df))
cut2 = int(0.85 * len(df))

train = df.iloc[:cut1]
val   = df.iloc[cut1:cut2]
test  = df.iloc[cut2:]

for name, part in [("train", train), ("val", val), ("test", test)]:
    print(f"{name}: {len(part)} rows, {part['arrival_date'].min().date()} to "
          f"{part['arrival_date'].max().date()}, cancel rate {part['is_canceled'].mean():.3f}")

train: 82994 rows, 2015-07-01 to 2017-02-11, cancel rate 0.363
val: 17784 rows, 2017-02-11 to 2017-05-22, cancel rate 0.392
test: 17785 rows, 2017-05-22 to 2017-08-31, cancel rate 0.397


In [6]:
# df["room_changed"] = (df["reserved_room_type"] != df["assigned_room_type"]).astype(int)

# for col in ["room_changed", "booking_changes", "days_in_waiting_list",
#             "required_car_parking_spaces", "total_of_special_requests"]:
#     summary = df.groupby(df[col] > 0)["is_canceled"].agg(Count="size", Cancel_Rate="mean")
#     summary["Cancel_Rate"] = (summary["Cancel_Rate"] * 100).round(2)
#     summary.index.name = f"{col} > 0"
#     display(summary)

train = train.copy()
train["room_changed"] = (train["reserved_room_type"] != train["assigned_room_type"]).astype(int)

for col in ["room_changed", "booking_changes", "days_in_waiting_list",
            "required_car_parking_spaces", "total_of_special_requests"]:
    summary = train.groupby(train[col] > 0)["is_canceled"].agg(Count="size", Cancel_Rate="mean")
    summary["Cancel_Rate"] = (summary["Cancel_Rate"] * 100).round(2)
    summary.index.name = f"{col} > 0"
    display(summary)

,Count,Cancel_Rate
room_changed > 0,,
False,71171,41.55
True,11823,4.79


,Count,Cancel_Rate
booking_changes > 0,,
False,70685,40.06
True,12309,14.79


,Count,Cancel_Rate
days_in_waiting_list > 0,,
False,79482,35.08
True,3512,64.18


,Count,Cancel_Rate
required_car_parking_spaces > 0,,
False,77457,38.91
True,5537,0.00


,Count,Cancel_Rate
total_of_special_requests > 0,,
False,51335,46.91
True,31659,19.14


In [7]:
drop_cols = [
    "assigned_room_type", "room_changed",      # assigned at check-in
    "required_car_parking_spaces",             # 0% cancellation → recorded at arrival
    "booking_date",                            # helper column, booking_month is kept
]
# keep arrival_date in the files so the next notebooks can see the time order

train = train.drop(columns=drop_cols, errors="ignore")
val   = val.drop(columns=drop_cols, errors="ignore")
test  = test.drop(columns=drop_cols, errors="ignore")

train.to_csv(data_dir / "train.csv", index=False)
val.to_csv(data_dir / "val.csv", index=False)
test.to_csv(data_dir / "test.csv", index=False)

print("Saved:", train.shape, val.shape, test.shape)



Saved: (82994, 36) (17784, 36) (17785, 36)


In [13]:
pd.set_option("display.max_columns", None)   # show all columns
pd.set_option("display.width", None)         # don't wrap columns onto new lines
display(train.head(20))

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,country,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,booking_changes,deposit_type,days_in_waiting_list,customer_type,adr,total_of_special_requests,has_agent,has_company,arrival_date,total_stay,total_guests,booking_lead_category,booking_category,historical_cancellation_rate,has_booking_history,booking_month
0,Resort Hotel,0,7,2015,July,27,1,0,1,1,0,0,BB,GBR,Direct,Direct,0,0,0,A,0,No Deposit,0,Transient,75.0,0,False,False,2015-07-01,1,1,short,individual,0.0,0,6
1,City Hotel,0,257,2015,July,27,1,0,2,1,0,0,HB,PRT,Offline TA/TO,TA/TO,0,0,0,A,1,No Deposit,0,Transient,80.0,0,True,False,2015-07-01,2,1,long,agent,0.0,0,10
2,City Hotel,0,257,2015,July,27,1,0,2,2,0,0,HB,PRT,Offline TA/TO,TA/TO,0,0,0,A,0,No Deposit,0,Transient,101.5,0,True,False,2015-07-01,2,2,long,agent,0.0,0,10
3,City Hotel,0,257,2015,July,27,1,0,2,2,0,0,HB,PRT,Offline TA/TO,TA/TO,0,0,0,A,0,No Deposit,0,Transient,101.5,0,True,False,2015-07-01,2,2,long,agent,0.0,0,10
4,City Hotel,0,257,2015,July,27,1,0,2,2,0,0,HB,PRT,Offline TA/TO,TA/TO,0,0,0,A,0,No Deposit,0,Transient,101.5,0,True,False,2015-07-01,2,2,long,agent,0.0,0,10
5,City Hotel,0,257,2015,July,27,1,0,2,2,0,0,HB,PRT,Offline TA/TO,TA/TO,0,0,0,A,0,No Deposit,0,Transient,101.5,0,True,False,2015-07-01,2,2,long,agent,0.0,0,10
6,City Hotel,0,257,2015,July,27,1,0,2,2,0,0,HB,PRT,Offline TA/TO,TA/TO,0,0,0,A,0,No Deposit,0,Transient,101.5,0,True,False,2015-07-01,2,2,long,agent,0.0,0,10
7,City Hotel,0,257,2015,July,27,1,0,2,2,0,0,HB,PRT,Offline TA/TO,TA/TO,0,0,0,A,0,No Deposit,0,Transient,101.5,0,True,False,2015-07-01,2,2,long,agent,0.0,0,10
8,City Hotel,0,257,2015,July,27,1,0,2,2,0,0,HB,PRT,Offline TA/TO,TA/TO,0,0,0,A,0,No Deposit,0,Transient,101.5,0,True,False,2015-07-01,2,2,long,agent,0.0,0,10
9,City Hotel,0,257,2015,July,27,1,0,2,1,0,0,HB,PRT,Offline TA/TO,TA/TO,0,0,0,A,1,No Deposit,0,Transient,80.0,0,True,False,2015-07-01,2,1,long,agent,0.0,0,10
